In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import defaultdict

def plot_clip_rewards(json_paths, figsize=(16, 8), save_path=None):
    """
    Plot CLIP rewards grouped by prompts across multiple experiments.
    
    Parameters:
    -----------
    json_paths : list of str
        List of paths to clip_rewards.json files
    figsize : tuple
        Figure size (width, height)
    save_path : str, optional
        Path to save the figure
    """
    # Prepare data structure
    all_data = {}
    
    # Read all JSON files
    for json_path in json_paths:
        # Extract experiment name
        path = Path(json_path)
        exp_name = path.parent.name
        
        # Read JSON
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        # Group scores by prompt and calculate mean
        prompt_scores = defaultdict(list)
        for prompt, score in zip(data['prompts'], data['all_scores']):
            prompt_scores[prompt].append(score)
        
        # Calculate mean for each prompt
        prompt_means = {prompt: np.mean(scores) 
                       for prompt, scores in prompt_scores.items()}
        
        all_data[exp_name] = prompt_means
    
    # Get all unique prompts (union across all experiments)
    all_prompts = sorted(set(prompt for exp_data in all_data.values() 
                            for prompt in exp_data.keys()))
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot each experiment
    colors = plt.cm.tab10(np.linspace(0, 1, len(all_data)))
    x_positions = np.arange(len(all_prompts))
    
    for idx, (exp_name, prompt_means) in enumerate(all_data.items()):
        # Get means for all prompts (0 if prompt not in this experiment)
        y_values = [prompt_means.get(prompt, np.nan) for prompt in all_prompts]
        
        ax.plot(x_positions, y_values, marker='o', linewidth=2, 
                markersize=6, label=exp_name, color=colors[idx], alpha=0.8)
    
    # Customize x-axis
    # Truncate long prompts for display
    display_labels = [prompt[:30] + '...' if len(prompt) > 30 else prompt 
                     for prompt in all_prompts]
    
    ax.set_xticks(x_positions)
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=9)
    
    # Labels and title
    ax.set_xlabel('Prompts', fontsize=12, fontweight='bold')
    ax.set_ylabel('Mean CLIP Reward', fontsize=12, fontweight='bold')
    ax.set_title('CLIP Rewards by Prompt Across Experiments', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Grid and legend
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(loc='best', framealpha=0.9, fontsize=10)
    
    # Tight layout
    plt.tight_layout()
    
    # Save if requested
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.show()
    
    # Print summary statistics
    print("\n=== Summary Statistics ===")
    for exp_name, prompt_means in all_data.items():
        values = list(prompt_means.values())
        print(f"\n{exp_name}:")
        print(f"  Number of prompts: {len(values)}")
        print(f"  Mean reward: {np.mean(values):.4f}")
        print(f"  Std reward: {np.std(values):.4f}")
        print(f"  Min reward: {np.min(values):.4f}")
        print(f"  Max reward: {np.max(values):.4f}")


# Alternative visualization: Heatmap style
def plot_clip_rewards_heatmap(json_paths, figsize=(14, 8), save_path=None):
    """
    Plot CLIP rewards as a heatmap for better visualization with many prompts.
    """
    # Prepare data
    all_data = {}
    
    for json_path in json_paths:
        path = Path(json_path)
        exp_name = path.parent.name
        
        with open(json_path, 'r') as f:
            data = json.load(f)
        
        prompt_scores = defaultdict(list)
        for prompt, score in zip(data['prompts'], data['all_scores']):
            prompt_scores[prompt].append(score)
        
        prompt_means = {prompt: np.mean(scores) 
                       for prompt, scores in prompt_scores.items()}
        
        all_data[exp_name] = prompt_means
    
    # Get all unique prompts
    all_prompts = sorted(set(prompt for exp_data in all_data.values() 
                            for prompt in exp_data.keys()))
    
    # Create matrix
    exp_names = list(all_data.keys())
    matrix = np.zeros((len(exp_names), len(all_prompts)))
    
    for i, exp_name in enumerate(exp_names):
        for j, prompt in enumerate(all_prompts):
            matrix[i, j] = all_data[exp_name].get(prompt, np.nan)
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=figsize)
    
    im = ax.imshow(matrix, aspect='auto', cmap='viridis', interpolation='nearest')
    
    # Set ticks
    ax.set_xticks(np.arange(len(all_prompts)))
    ax.set_yticks(np.arange(len(exp_names)))
    
    # Labels
    display_labels = [prompt[:40] + '...' if len(prompt) > 40 else prompt 
                     for prompt in all_prompts]
    ax.set_xticklabels(display_labels, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(exp_names, fontsize=10)
    
    # Colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Mean CLIP Reward', rotation=270, labelpad=20, fontsize=11)
    
    # Title
    ax.set_title('CLIP Rewards Heatmap: Experiments vs Prompts', 
                 fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"Heatmap saved to {save_path}")
    
    plt.show()


# Example usage:
if __name__ == "__main__":
    # Example paths
    json_paths = [
        "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/only_5_steps/clip_rewards.json",#1.2845
        "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/another_only_5_steps/clip_rewards.json", #1.2851
        "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/b2diffu_try2/clip_rewards.json", #1.2858
        "/media/ajad/YourBook/AshokSaugatResearchBackup/AshokSaugatResearch/b2diff/outputs/fk_steering/clip_rewards.json", #1.2701
    ]
    
    # Line plot
    plot_clip_rewards(json_paths, save_path="clip_rewards_comparison.png")
    
    # Heatmap (better for many prompts)
    plot_clip_rewards_heatmap(json_paths, save_path="clip_rewards_heatmap.png")